In [11]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda Distribuida Anti-Bloqueos: Portal Inmobiliario
# ==============================================================================

import os
import time
import re
import random #ANTI-BOT: Importamos la librería para simular al humano
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 2
PAGINA_FIN = 2
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"Entorno virtual listo. Asignación: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (CON PAGINACIÓN DINÁMICA)
    # ==============================================================================
    url_base = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"

    # El ciclo ahora respeta los límites que definieron arriba
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
        
        # Cálculo de salto de página de Portal Inmobiliario
        if pagina_actual == 1:
            url_pagina = url_base
        else:
            desde = ((pagina_actual - 1) * 48) + 1
            url_pagina = f"{url_base}_Desde_{desde}_NoIndex_True"
            
        driver.get(url_pagina)

        try:
            WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
            
            # ANTI-BOT: Scroll con ritmo humano y cantidad aleatoria
            for _ in range(random.randint(3, 5)):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(random.uniform(0.8, 1.5))
                
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(random.uniform(1.5, 3.0)) 

            bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
            
            for bloque in bloques:
                try:
                    # 1. URL y Título
                    url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                    titulo = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent").strip()
                    
                    # 2. Extracción y Normalización de Ubicación (Prioridad Coquimbo)
                    ubicacion_cruda = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent").strip()
                    ubi_minuscula = ubicacion_cruda.lower()
                    
                    if "coquimbo" in ubi_minuscula:
                        ubicacion = "Coquimbo"      
                    elif "la serena" in ubi_minuscula:
                        ubicacion = "La Serena"     
                    else:
                        continue 

                    # 3. Extracción de Precio
                    try:
                        p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                    except:
                        p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")
                    
                    v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                    precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                    if url != "Sin URL" and precio > 0:
                        catalogo_urls.append({
                            "url": url,
                            "titulo": titulo,
                            "ubicacion": ubicacion,
                            "precio": precio
                        })
                except:
                    continue
        except Exception as e:
            print(f"🚨 Advertencia en página {pagina_actual}: {e}")
            
        # ANTI-BOT: Pausa larga antes de saltar de página en el catálogo
        time.sleep(random.uniform(3.0, 5.5))

    print(f"\nCatálogo listo: {len(catalogo_urls)} propiedades encontradas. Recolectando amenidades...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"[{i+1}/{len(catalogo_urls)}] Inspeccionando: {propiedad['titulo'][:30]}...")
        
        registro = {
            "responsable": "Millaray Zalazar", # Aquí cambiar el nombre de cada uno
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": propiedad["titulo"],
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": propiedad["precio"],
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-collapsable__container"))
            )
            
            filas_tabla = driver.find_elements(By.CSS_SELECTOR, ".andes-table__row")
            
            for fila in filas_tabla:
                texto_fila = fila.get_attribute("textContent").lower()
                
                # Extracción Numérica
                if "superficie total" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["m2"] = int(nums[0])
                        
                elif "dormitorios" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["dormitorios"] = int(nums[0])
                        
                elif "baños" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums: registro["baños"] = int(nums[0])
                        
                elif "estacionamiento" in texto_fila:
                    nums = re.findall(r'\d+', texto_fila)
                    if nums and int(nums[0]) > 0:
                        registro["estacionamiento"] = int(nums[0])
                    elif "sí" in texto_fila or "si" in texto_fila:
                        registro["estacionamiento"] = 1
                
                # Extracción Booleana (1 o 0)
                elif "piscina" in texto_fila and "sí" in texto_fila:
                    registro["piscina"] = 1
                elif "quincho" in texto_fila and "sí" in texto_fila:
                    registro["quincho"] = 1
                elif "terraza" in texto_fila and "sí" in texto_fila:
                    registro["terraza"] = 1
                elif "gimnasio" in texto_fila and "sí" in texto_fila:
                    registro["gimnasio"] = 1
                elif "lavander" in texto_fila and "sí" in texto_fila: 
                    registro["lavanderia"] = 1

            datos_finales.append(registro)

        except Exception as e:
            print(f"Tabla no encontrada o página caída. Guardando con valores base.")
            registro["precio"] = 0.0
            datos_finales.append(registro)
            
        # ANTI-BOT
        driver.delete_all_cookies() 
        time.sleep(random.uniform(2.8, 5.2)) 

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()
        print("\nNavegador cerrado.")

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- Proceso de guardado ---")
try:
    if datos_finales:
        # 🛡️ Ajuste a "localhost" para evitar el Errno -5
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos = 0
        registros_actualizados = 0

        for d in datos_finales:
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )

            if resultado.upserted_id:
                registros_nuevos += 1
            else:
                registros_actualizados += 1

        print(f"{datos_finales[0]['responsable']}")
        print(f"   -> {registros_nuevos} departamentos creados.")
        print(f"   -> {registros_actualizados} departamentos actualizados.")
    else:
        print("No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"Error conectando a la Base de Datos: {e}")

Entorno virtual listo. Asignación: Páginas 2 a la 2.

--- [FASE 1] Recolectando enlaces de la Página 2 ---

Catálogo listo: 46 propiedades encontradas. Recolectando amenidades...
[1/46] Inspeccionando: Departamento En Arriendo Cond ...
Tabla no encontrada o página caída. Guardando con valores base.
[2/46] Inspeccionando: Arriendo Departamento Sector L...
Tabla no encontrada o página caída. Guardando con valores base.
[3/46] Inspeccionando: Acogedor Departamento Con  Acc...
Tabla no encontrada o página caída. Guardando con valores base.
[4/46] Inspeccionando: Se Arrienda Departamento En La...
[5/46] Inspeccionando: Arriendo Año Corrido 1d-1b-1 E...
[6/46] Inspeccionando: Año Corrido, Marina Sol, Cerca...
[7/46] Inspeccionando: Arriendo Departamento En Aveni...
[8/46] Inspeccionando: Departamento A Estrenar, Arrie...
[9/46] Inspeccionando: Depto Nuevo Estilo Mariposa En...
[10/46] Inspeccionando: Depto. Amoblado Bosque San Car...
[11/46] Inspeccionando: Arriendo Departamento Amoblado...


In [3]:
from pyspark.sql.functions import col, avg, count, round, format_number, concat, lit

# Llenamos los nulos iniciales de las columnas que ya vienen limpias del scraper
df_limpio = df.na.fill({
    "dormitorios": 0,
    "baños": 0,
    "m2": 0,
    "precio": 0,
    "estacionamiento": 0,
    "piscina": 0,
    "quincho": 0,
    "terraza": 0,
    "gimnasio": 0,
    "lavanderia": 0
})

df_validado = df_limpio.filter(col("precio") > 10000)

print("Reporte por dormitorios")
reporte_dorms = df_validado.groupBy("dormitorios").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
     "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("dormitorios")

reporte_dorms.select("dormitorios","cantidad_propiedades","precio_promedio").show()


print("Reporte por baños")
reporte_baños = df_validado.groupBy("baños").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("baños")

reporte_baños.select("baños","cantidad_propiedades","precio_promedio").show()


print("Reporte por estacionamiento (0=No, 1=Sí)")
reporte_estac = df_validado.groupBy("estacionamiento").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("estacionamiento")

reporte_estac.select("estacionamiento","cantidad_propiedades","precio_promedio").show()


print("Reporte por piscina (0=No, 1=Sí)")
reporte_piscina = df_validado.groupBy("piscina").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("piscina")

reporte_piscina.select("piscina","cantidad_propiedades","precio_promedio").show()


print("Reporte por quincho (0=No, 1=Sí)")
reporte_quincho = df_validado.groupBy("quincho").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("quincho")

reporte_quincho.select("quincho","cantidad_propiedades","precio_promedio").show()


print("Reporte por terraza (0=No, 1=Sí)")
reporte_terraza = df_validado.groupBy("terraza").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("terraza")

reporte_terraza.select("terraza","cantidad_propiedades","precio_promedio").show()


print("Reporte por gimnasio (0=No, 1=Sí)")
reporte_gimnasio = df_validado.groupBy("gimnasio").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("gimnasio")

reporte_gimnasio.select("gimnasio","cantidad_propiedades","precio_promedio").show()


print("Reporte por lavanderia (0=No, 1=Sí)")
reporte_lavanderia = df_validado.groupBy("lavanderia").agg(
        count("*").alias("cantidad_propiedades"),
        avg("precio").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"),0),0))
).orderBy("lavanderia")

reporte_lavanderia.select("lavanderia","cantidad_propiedades","precio_promedio").show()

Reporte por dormitorios
+-----------+--------------------+---------------+
|dormitorios|cantidad_propiedades|precio_promedio|
+-----------+--------------------+---------------+
|          0|                   4|       $600,000|
|          1|                   6|       $435,000|
|          2|                  42|       $581,786|
|          3|                  24|       $625,000|
|          4|                   4|     $1,612,500|
+-----------+--------------------+---------------+

Reporte por baños
+-----+--------------------+---------------+
|baños|cantidad_propiedades|precio_promedio|
+-----+--------------------+---------------+
|    0|                   4|       $600,000|
|    1|                  30|       $485,500|
|    2|                  42|       $654,286|
|    3|                   3|     $1,733,333|
|    4|                   1|     $1,250,000|
+-----+--------------------+---------------+

Reporte por estacionamiento (0=No, 1=Sí)
+---------------+--------------------+-------------

In [2]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: 
    spark.stop()
except: 
    pass

# Iniciamos la sesión conectando a la base de datos correcta
spark = SparkSession.builder \
    .appName("Analisis_BigData_RealEstate") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/Prueba.Pruebaaa") \
    .getOrCreate()

# Cargamos los datos extraídos en el DataFrame
df = spark.read.format("mongodb").load()

print(f"Éxito total: {df.count()} departamentos cargados desde MongoDB.")
df.show(5)

Éxito total: 94 departamentos cargados desde MongoDB.
+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+---------+-------+----------------+-------+--------------------+---------+--------------------+
|                 _id|baños|dormitorios|estacionamiento|   fecha_extraccion|gimnasio|lavanderia| m2|piscina|   precio|quincho|     responsable|terraza|              titulo|ubicacion|                 url|
+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+---------+-------+----------------+-------+--------------------+---------+--------------------+
|69e8203aab0547cd5...|    0|          0|              0|2026-04-22 01:05:26|       0|         0|  0|      0| 650000.0|      0|Millaray Zalazar|      0|Se Arrienda Depar...| Coquimbo|https://portalinm...|
|69e8203aab0547cd5...|    1|          1|              1|2026-04-22 01:05:42|       1|         1| 43|      0| 450000.0|      0|Mill

In [2]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Código para Portal Inmobiliario
# ==============================================================================

import os
import time
import re
import random 
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 6
PAGINA_FIN = 6
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"Entorno virtual listo. Asignación: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

# ANTI-BOT Nivel 1: Apagar las banderas de automatización de Chrome
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    
    # ANTI-BOT Nivel 2: Inyectar código en la página antes de que cargue 
    # para borrar la palabra "webdriver" del navegador y parecer un humano.
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    
    url_base = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"

    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO
    # ==============================================================================
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- [FASE 1] Recolectando enlaces de la Página {pagina_actual} ---")
        
        if pagina_actual == 1:
            url_pagina = url_base
        else:
            desde = ((pagina_actual - 1) * 48) + 1
            url_pagina = f"{url_base}_Desde_{desde}_NoIndex_True"
            
        driver.get(url_pagina)

        try:
            WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item")))
            
            # Scroll Ultra Lento y Suave
            for _ in range(random.randint(12, 18)): 
                driver.execute_script("window.scrollBy(0, 350);") 
                time.sleep(random.uniform(0.3, 0.7)) 
                
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(random.uniform(1.5, 2.2)) 

            bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
            
            for bloque in bloques:
                try:
                    url = bloque.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[0].split("?")[0]
                    titulo = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent").strip()
                    
                    ubicacion_cruda = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent").strip()
                    ubi_minuscula = ubicacion_cruda.lower()
                    
                    if "coquimbo" in ubi_minuscula: ubicacion = "Coquimbo"      
                    elif "la serena" in ubi_minuscula: ubicacion = "La Serena"     
                    else: continue 

                    try:
                        p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                    except:
                        p_text = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")
                    
                    v_limpio = p_text.replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
                    precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                    if url != "Sin URL" and precio > 0:
                        catalogo_urls.append({
                            "url": url,
                            "titulo": titulo,
                            "ubicacion": ubicacion,
                            "precio": precio
                        })
                except: continue
                
        except Exception as e:
            print(f"Advertencia en página {pagina_actual}: {e}")
            
        time.sleep(random.uniform(3.0, 5.5))

    print(f"\nCatálogo listo: {len(catalogo_urls)} propiedades encontradas. Recolectando amenidades...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA POR PROPIEDAD
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        print(f"[{i+1}/{len(catalogo_urls)}] Inspeccionando: {propiedad['titulo'][:30]}...")
        
        registro = {
            "responsable": "Millaray Zalazar", 
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": propiedad["titulo"],
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": propiedad["precio"], # Valor original del catálogo
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"] 
        }

        try:
            driver.get(propiedad["url"])
            
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            texto_pagina = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()
            
            # Lector de Letreros (Detecta publicaciones falsas o arrendadas)
            if "publicación pausada" in texto_pagina or "publicación finaliz" in texto_pagina:
                print("   -> Publicación inactiva. Anulando precio a 0.0")
                registro["precio"] = 0.0
                datos_finales.append(registro)
            else:
                # Si está activa, leemos la tabla
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-collapsable__container")))
                filas_tabla = driver.find_elements(By.CSS_SELECTOR, ".andes-table__row")
                
                for fila in filas_tabla:
                    texto_fila = fila.get_attribute("textContent").lower()
                    
                    if "superficie total" in texto_fila:
                        nums = re.findall(r'\d+', texto_fila)
                        if nums: registro["m2"] = int(nums[0])
                    elif "dormitorios" in texto_fila:
                        nums = re.findall(r'\d+', texto_fila)
                        if nums: registro["dormitorios"] = int(nums[0])
                    elif "baños" in texto_fila:
                        nums = re.findall(r'\d+', texto_fila)
                        if nums: registro["baños"] = int(nums[0])
                    elif "estacionamiento" in texto_fila:
                        nums = re.findall(r'\d+', texto_fila)
                        if nums and int(nums[0]) > 0:
                            registro["estacionamiento"] = int(nums[0])
                        elif "sí" in texto_fila or "si" in texto_fila:
                            registro["estacionamiento"] = 1
                    
                    elif "piscina" in texto_fila and "sí" in texto_fila: registro["piscina"] = 1
                    elif ("quincho" in texto_fila and "sí" in texto_fila) or ("parrilla" in texto_fila and "sí" in texto_fila): registro["quincho"] = 1
                    elif "terraza" in texto_fila and "sí" in texto_fila: registro["terraza"] = 1
                    elif "gimnasio" in texto_fila and "sí" in texto_fila: registro["gimnasio"] = 1
                    elif "lavander" in texto_fila and "sí" in texto_fila: registro["lavanderia"] = 1

                datos_finales.append(registro)

        except Exception as e:
            # Si hay Error 404 o falla total de la página, también anulamos el precio
            print(f"   -> Página caída o tabla no encontrada. Anulando precio a 0.0")
            registro["precio"] = 0.0
            datos_finales.append(registro)
            
        driver.delete_all_cookies() 
        time.sleep(random.uniform(2.8, 5.2)) 

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()
        print("\nNavegador cerrado.")

# ==============================================================================
# FASE 3: GUARDADO EN MONGODB (UPSERT)
# ==============================================================================
print("\n--- Proceso de guardado ---")
try:
    if datos_finales:
        client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba"]
        coleccion = db["Pruebaaa"] 

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            resultado = coleccion.update_one(
                {"url": d["url"]}, 
                {"$set": d},       
                upsert=True        
            )
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"{datos_finales[0]['responsable']}")
        print(f"   -> {registros_nuevos} departamentos creados.")
        print(f"   -> {registros_actualizados} departamentos actualizados.")
    else:
        print("No se encontró ningún dato para guardar.")

except Exception as e:
    print(f"Error conectando a la Base de Datos: {e}")

Entorno virtual listo. Asignación: Páginas 6 a la 6.

--- [FASE 1] Recolectando enlaces de la Página 6 ---

Catálogo listo: 33 propiedades encontradas. Recolectando amenidades...
[1/33] Inspeccionando: Departamento María Angélica Id...
[2/33] Inspeccionando: Departamento En Arriendo En Av...
[3/33] Inspeccionando: Excelente Departamento Condomi...
[4/33] Inspeccionando: Hermoso Depto 3d&#43;2b&#43;e&...
[5/33] Inspeccionando: Arriendo Departamento  3 Dorm,...
[6/33] Inspeccionando: Departamento En Arriendo De 2 ...
[7/33] Inspeccionando: Departamento Manuel Jesús Rive...
[8/33] Inspeccionando: Depto Moderno - Entrega Inmedi...
[9/33] Inspeccionando: Estudio Un Ambiente Cercano A ...
[10/33] Inspeccionando: Se Arrienda Depto Año Corrido,...
[11/33] Inspeccionando: Arriendo Depto Nuevo 1d/1b A P...
[12/33] Inspeccionando: Arriendo Departamento Amoblado...
[13/33] Inspeccionando: Se Arrienda Depto En Coquimbo,...
[14/33] Inspeccionando: Departamento Full Equipado, Id...
[15/33] Inspeccion